# Allocation supplémentaire de 10 MD

Ce notebook analyse l'impact d'une **enveloppe additionnelle de 10 MD** investie **uniquement dans la poche optimisable** du portefeuille Maghrebia, en testant l'objectif **ROE cible = TSR + 4 %** sur les trois scénarios APT (Prudent, Central, Optimiste).

Le notebook s'appuie sur les sorties méthodologiques validées des notebooks 01 (diagnostic, APT, scénarios) et 02 (optimisation). Max Sharpe est volontairement exclu, conformément aux choix méthodologiques du notebook 02.

## Limite méthodologique : ROE proxy

L'objectif `ROE = TSR + 4 %` est interprété ici comme une **cible de rendement financier attendu**. Cette approximation ne remplace pas le ROE comptable, qui dépend également du résultat technique, des charges, de la réassurance, de la fiscalité, des provisions et des fonds propres. Le ROE comptable réel est :

$$ROE = \frac{Resultat\ net}{Fonds\ propres}$$

Dans ce notebook, faute d'un modèle complet du compte de résultat et du bilan, on utilise comme proxy le rendement financier attendu du portefeuille. L'hypothèse TSR est lue depuis le notebook 01 et doit rester synchronisée avec le notebook 02.

In [14]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from maghrebia_quant.allocation_10md import (
    AdditionalAllocationConfig,
    SCENARIO_ORDER,
    export_multi_scenario_analysis,
    run_multi_scenario_allocation,
)

ADDITIONAL_BUDGET = 10_000_000
ROE_SPREAD_TARGET = 0.04

CONFIG = AdditionalAllocationConfig(
    additional_budget=ADDITIONAL_BUDGET,
    roe_spread_target=ROE_SPREAD_TARGET,
    monte_carlo_portfolios=20_000,
)

EXPORT_DIR = PROJECT_ROOT / "exports"
FIGURE_DIR = EXPORT_DIR / "figures" / "notebook_03"
EXCEL_PATH = EXPORT_DIR / "notebook_03_allocation_10MD_results.xlsx"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

pd.options.display.float_format = "{:,.6f}".format
pd.options.display.max_columns = 60

ImportError: cannot import name 'SCENARIO_ORDER' from 'maghrebia_quant.allocation_10md' (C:\Users\Houssem\OneDrive\Desktop\Projet\JupyterProject1\src\maghrebia_quant\allocation_10md.py)

In [ ]:
result = run_multi_scenario_allocation(PROJECT_ROOT, CONFIG)

TSR = result["tsr"]
TARGET_RETURN = result["by_scenario"]["APT_Central"]["state"]["TARGET_RETURN"]
V_TOTAL_CURRENT = result["V_TOTAL_CURRENT"]
V_OPT_CURRENT = result["V_OPT_CURRENT"]
V_FIXED_CURRENT = result["V_FIXED_CURRENT"]
V_OPT_FINAL = V_OPT_CURRENT + ADDITIONAL_BUDGET
V_TOTAL_FINAL = V_TOTAL_CURRENT + ADDITIONAL_BUDGET

assert abs(V_TOTAL_CURRENT - (V_OPT_CURRENT + V_FIXED_CURRENT)) <= 1e-2
assert abs(V_OPT_FINAL - (V_OPT_CURRENT + ADDITIONAL_BUDGET)) <= 1e-8
assert abs(V_TOTAL_FINAL - (V_TOTAL_CURRENT + ADDITIONAL_BUDGET)) <= 1e-8

display(Markdown(
    f"**TSR retenu (depuis notebook 01) :** {TSR:.2%}  \n"
    f"**TARGET_RETURN = TSR + 4 % :** {TARGET_RETURN:.2%}  \n"
    f"**V_TOTAL_CURRENT :** {V_TOTAL_CURRENT:,.0f} TND  \n"
    f"**V_OPT_CURRENT :** {V_OPT_CURRENT:,.0f} TND  \n"
    f"**V_FIXED_CURRENT :** {V_FIXED_CURRENT:,.0f} TND  \n"
    f"**ADDITIONAL_BUDGET :** {ADDITIONAL_BUDGET:,.0f} TND  \n"
    f"**V_OPT_FINAL :** {V_OPT_FINAL:,.0f} TND  \n"
    f"**V_TOTAL_FINAL :** {V_TOTAL_FINAL:,.0f} TND"
))

## 1. Cadrage méthodologique

Les 10 MD ne sont **pas** ajoutés au portefeuille global comme univers d'allocation. Ils sont ajoutés **uniquement à la poche optimisable**, qui comprend :

- titres de l'État ;
- emprunts obligataires / obligations corporate ;
- actions cotées.

Les actifs non optimisables restent figés : immobilier, OPCVM non traités, SICAR/SICAF, actions non cotées, participations, placements non modélisés.

In [ ]:
display(result["01_Hypotheses"])
display(result["02_Current_Portfolio"])

## 2. Univers optimisable

Univers d'investissement, métriques par actif et statut qualité APT. Tous les actifs sont inclus dans l'optimisation et appartiennent à la poche optimisable.

In [ ]:
pocket = result["03_Optimizable_Pocket"]
display(pocket)

assert pocket["Included_in_Optimization"].all()
universe_assets = set(result["universe"]["asset_id"])
assert set(pocket["Asset"]).issubset(universe_assets)

## 3. Trois scénarios APT

Les trois scénarios APT issus du notebook 01 permettent de tester si l'allocation additionnelle de 10 MD reste robuste en environnement prudent, central et optimiste. Les deltas par rapport au scénario central sont affichés pour repérer les actifs les plus sensibles aux hypothèses macro.

In [ ]:
apt = result["04_APT_Scenarios"]
display(apt)

display(Markdown(
    "_Les trois scénarios APT permettent de tester si l'allocation additionnelle de 10 MD reste "
    "robuste en environnement prudent, central et optimiste. Le scénario central reste la "
    "référence principale ; les scénarios prudent et optimiste servent de bornes._"
))

## 4. Rendements actuels par scénario

Pour chaque scénario APT, on calcule :

- `R_OPT_CURRENT` = somme des poids actuels de la poche optimisable × rendements APT du scénario ;
- `R_TOTAL_CURRENT_OR_PROXY` = proxy total avec la poche figée à rendement nul.

Il n'y a pas de mélange entre rendement historique et rendement APT ; la référence est le rendement APT.

In [ ]:
display(result["05b_Current_Returns"])

## 5. Rendement requis sur les 10 MD

Rendement marginal requis pour atteindre la cible `TSR + 4 %` sur chaque scénario :

- sur la poche optimisable : `R_REQUIRED = (TARGET × V_OPT_FINAL − R_OPT_CURRENT × V_OPT_CURRENT) / ADDITIONAL_BUDGET` ;
- sur le portefeuille total : formule analogue.

Le `Best_Achievable_R_Additional` est le rendement maximal atteignable sur les 10 MD sous contraintes.

In [ ]:
display(result["05_Required_Returns"])

any_infeasible = result["05_Required_Returns"][["Feasibility_Status_Opt", "Feasibility_Status_Total"]].eq("INFEASIBLE").any().any()
if any_infeasible:
    display(Markdown(
        "**TARGET_NOT_REACHED** sous au moins un scénario : le rendement marginal requis sur les 10 MD "
        "est supérieur aux rendements réalistes disponibles dans l'univers d'investissement. "
        "Le résultat traduit un effet de taille de l'enveloppe et non un échec du solveur."
    ))

## 6. Résultats par modèle et par scénario

Pour chaque scénario APT, les modèles d'allocation suivants sont exécutés (Max Sharpe volontairement exclu) :

1. **Prorata** de la poche optimisable actuelle (benchmark) ;
2. **Minimum Variance** ;
3. **Mean-Variance** avec trois niveaux d'aversion (λ = 2, 5, 10) ;
4. **Maximum Return** sous contraintes (borne supérieure agressive, **pas** recommandation finale) ;
5. **Mean-CVaR 95 %** ;
6. **Risk Parity** ;
7. **Monte Carlo** : Max Return, Min Volatility, Min CVaR, Best Scoring ;
8. **Scoring multicritère final** (rendement, volatilité, CVaR, diversification, conformité, etc.).

Maximum Return est présenté comme **borne supérieure** ; la recommandation finale provient du scoring.

In [ ]:
cross = result["07_Model_Results_By_Scenario"]
display(cross)

### Impact sur la poche optimisable et sur le portefeuille global

In [ ]:
display(result["08_Impact_Optimizable_Pocket"])
display(result["09_Impact_Total_Portfolio"])

## 7. Allocations détaillées des 10 MD par modèle

Pour chaque modèle et chaque scénario, on affiche les actifs ayant reçu une allocation significative (≥ 0.01 %). Tous les actifs alloués appartiennent à la poche optimisable.

In [ ]:
alloc = result["06_Allocations_10MD"]
alloc_display = alloc.loc[alloc["display_in_main_tables"]].copy()
display(alloc_display[[
    "Scenario", "Model", "asset_name", "asset_class", "weight_10md",
    "amount_allocated_DT", "final_weight_opt", "final_weight_total",
]].head(80))

assert (alloc["weight_10md"] >= -1e-10).all()
for (scenario, model), group in alloc.groupby(["Scenario", "Model"]):
    if model.startswith("Monte_Carlo_") and model not in result["by_scenario"][scenario]["results_models"]["Model"].tolist():
        continue
    s = float(group["weight_10md"].sum())
    assert abs(s - 1.0) <= 1e-6, f"{scenario}/{model}: weights sum = {s}"
    total_dt = float(group["amount_allocated_DT"].sum())
    assert abs(total_dt - ADDITIONAL_BUDGET) <= 1.0, f"{scenario}/{model}: amount = {total_dt}"
    asset_classes = set(group.loc[group["weight_10md"] > 0, "asset_type"])
    assert asset_classes.issubset({"government_bond", "corporate_bond", "listed_equity"}), \
        f"Actif non optimisable détecté pour {scenario}/{model}"

## 8. Recommandation finale par scénario

Le scoring multicritère choisit, pour chaque scénario APT, l'allocation institutionnellement défendable. Maximum Return est conservé comme borne supérieure mais pénalisé par le scoring.

In [ ]:
display(result["14_Final_Recommendation"])
display(result["14b_Status"])

## 9. Contraintes réglementaires sur le portefeuille global final

Les contraintes d'optimisation s'appliquent à l'allocation des 10 MD, mais les contraintes réglementaires sont vérifiées sur le **portefeuille global final** (portefeuille actuel + allocation des 10 MD). Statuts : `PASSED`, `FAILED`, `NON_TESTABLE_DATA_MISSING`.

In [ ]:
central_reco = result["by_scenario"]["APT_Central"]["recommended_model"]
regulatory = result["10_Regulatory_Checks"]
reg_central = regulatory.loc[
    regulatory["Scenario"].eq("APT_Central") & regulatory["model"].eq(central_reco)
]
display(reg_central[[
    "Scenario", "model", "Constraint", "Status", "exposition_avant",
    "exposition_apres", "seuil", "marge_apres", "testable",
]])

testable_central = reg_central.loc[reg_central["testable"]]
assert not testable_central["Status"].eq("FAILED").any()

display(Markdown(
    "Aucune violation n'est détectée sur les contraintes testables. Certaines contraintes "
    "réglementaires restent non testables faute de données détaillées (référentiel émetteur, "
    "capital social, détail OPCVM, actions non cotées)."
))

## 10. Monte Carlo (sans Max Sharpe)

Sélection Monte Carlo distincte pour chaque scénario : Max Return, Min Volatility, Min CVaR, Best Scoring.

In [ ]:
mc_selected = result["11b_Monte_Carlo_Selected"]
display(mc_selected[[
    "Scenario", "Selection", "portfolio_id", "R_additional", "R_opt_final",
    "R_total_final", "Volatility_additional", "CVaR_95", "Target_Status",
    "Selection_Note",
]])

for scenario in SCENARIO_ORDER:
    assert (result["by_scenario"][scenario]["monte_carlo"].shape[0] >= 15_000)

### VaR / CVaR / Sharpe — précautions

Le ratio de Sharpe est affiché à titre indicatif. Il **n'est pas utilisé comme critère principal de décision**, compte tenu du lissage possible de la volatilité sur certains actifs obligataires revalorisés par modèle.

Les VaR/CVaR à 95 % sont calculées sur les **rendements périodiques** (jamais sur des rendements déjà annualisés). Une VaR ou une CVaR proche de zéro pour un portefeuille contenant des actions doit être interprétée avec prudence : elle peut refléter une période courte ou un lissage des séries.

## 11. Analyse de sensibilité

Deux questions complémentaires :

1. **Rendement final** si les 10 MD rapportent 8 %, 10 %, 12 %, 15 %, 20 %, 30 %, 40 %, 50 %, 60 % ;
2. **Budget additionnel requis** pour atteindre la cible si le rendement additionnel réaliste est 10 %, 12 %, 15 %, 20 %.

Si le rendement additionnel hypothétique est inférieur ou égal à la cible, la formule est non définie.

In [ ]:
sensitivity = result["13_Sensitivity_Analysis"]
display(sensitivity)

## 12. Warnings qualité

In [ ]:
display(result["Warnings_Quality"])

## 13. Graphiques

Dix graphiques utiles produits sous Plotly, exportés en HTML dans `exports/figures/notebook_03/`.

In [ ]:
exported_figures = []

def show_and_export_fig(fig, name, output_dir=FIGURE_DIR):
    fig.show(renderer="notebook_connected")
    path = output_dir / f"{name}.html"
    fig.write_html(path, include_plotlyjs="cdn", full_html=True)
    exported_figures.append(path)
    return path

for old in FIGURE_DIR.glob("*.html"):
    old.unlink()

### Graphique 1 — Composition actuelle de la poche optimisable

Mise en évidence des poids actuels par classe d'actif et par titre.

In [ ]:
fig = px.bar(
    result["03_Optimizable_Pocket"],
    x="Asset", y="Poids actuel poche optimisable", color="Classe",
    title="Composition actuelle de la poche optimisable",
    labels={"Asset": "Actif", "Poids actuel poche optimisable": "Poids"},
)
fig.update_layout(xaxis_tickangle=-45)
show_and_export_fig(fig, "01_composition_poche_optimisable")

### Graphique 2 — Allocation des 10 MD par modèle recommandé et par scénario

Pour chaque scénario APT, on affiche la répartition des 10 MD selon le modèle recommandé.

In [ ]:
rows = []
for scenario in SCENARIO_ORDER:
    scn = result["by_scenario"][scenario]
    model = scn["recommended_model"]
    sub = result["06_Allocations_10MD"]
    sub = sub.loc[
        sub["Scenario"].eq(scenario)
        & sub["Model"].eq(model)
        & sub["display_in_main_tables"]
    ]
    rows.append(sub)
alloc_reco = pd.concat(rows, ignore_index=True)
fig = px.bar(
    alloc_reco, x="asset_name", y="amount_allocated_DT", color="Scenario", barmode="group",
    title="Allocation des 10 MD — modèle recommandé par scénario",
    labels={"asset_name": "Actif", "amount_allocated_DT": "Montant alloué (TND)"},
)
fig.update_layout(xaxis_tickangle=-45)
show_and_export_fig(fig, "02_allocation_10md_par_scenario")

### Graphique 3 — Rendement additionnel des 10 MD par modèle et scénario

In [ ]:
fig = px.bar(
    cross, x="Model", y="R_additional", color="Scenario", barmode="group",
    title="Rendement additionnel des 10 MD par modèle et scénario",
    labels={"Model": "Modèle", "R_additional": "Rendement additionnel"},
)
fig.update_layout(xaxis_tickangle=-35)
show_and_export_fig(fig, "03_rendement_additionnel_par_modele_scenario")

### Graphique 4 — Rendement final de la poche optimisable vs cible

In [ ]:
fig = px.bar(
    cross, x="Model", y="R_opt_final", color="Scenario", barmode="group",
    title="Rendement final de la poche optimisable vs cible",
)
fig.add_hline(y=TARGET_RETURN, line_dash="dash", annotation_text="Cible")
fig.update_layout(xaxis_tickangle=-35)
show_and_export_fig(fig, "04_rendement_final_poche_vs_cible")

### Graphique 5 — Rendement final du portefeuille total vs cible

In [ ]:
fig = px.bar(
    cross, x="Model", y="R_total_final", color="Scenario", barmode="group",
    title="Rendement final du portefeuille total vs cible",
)
fig.add_hline(y=TARGET_RETURN, line_dash="dash", annotation_text="Cible")
fig.update_layout(xaxis_tickangle=-35)
show_and_export_fig(fig, "05_rendement_final_total_vs_cible")

### Graphique 6 — Écart à la cible par scénario et modèle

In [ ]:
gap = cross.melt(
    id_vars=["Scenario", "Model"], value_vars=["Gap_Opt", "Gap_Total"],
    var_name="Périmètre", value_name="Gap",
)
fig = px.bar(
    gap, x="Model", y="Gap", color="Scenario", barmode="group", facet_row="Périmètre",
    title="Écart à la cible par scénario et modèle (Gap = R_final − Cible)",
)
fig.update_layout(xaxis_tickangle=-35, height=620)
show_and_export_fig(fig, "06_gap_a_la_cible")

### Graphique 7 — Nuage Monte Carlo (sans Max Sharpe) par scénario

In [ ]:
mc = result["11_Monte_Carlo"].copy()
selected_ids = result["11b_Monte_Carlo_Selected"][["Scenario", "portfolio_id"]].dropna()
selected_ids["portfolio_id"] = selected_ids["portfolio_id"].astype(int)
selected_set = set(zip(selected_ids["Scenario"], selected_ids["portfolio_id"]))
mc["Selected"] = [
    (s, int(p)) in selected_set for s, p in zip(mc["Scenario"], mc["portfolio_id"])
]
fig = px.scatter(
    mc, x="Volatility_additional", y="R_additional",
    color="Scenario", symbol="Selected", opacity=0.35,
    title="Monte Carlo — rendement vs volatilité (10 MD), portefeuilles sélectionnés mis en avant",
)
fig.add_hline(y=TARGET_RETURN, line_dash="dash", annotation_text="Cible TSR+4%")
show_and_export_fig(fig, "07_monte_carlo_par_scenario")

### Graphique 8 — Risque vs rendement des modèles déterministes

In [ ]:
deterministic_models = [
    "Prorata_Current_Optimizable_Pocket", "Minimum_Variance",
    "Mean_Variance_Aversion_2", "Mean_Variance_Aversion_5", "Mean_Variance_Aversion_10",
    "Max_Return_Constraints", "Mean_CVaR", "Risk_Parity",
]
det = cross.loc[cross["Model"].isin(deterministic_models)]
fig = px.scatter(
    det, x="Volatility_additional", y="R_additional", color="Scenario", symbol="Model",
    text="Model",
    title="Risque vs rendement des modèles déterministes (10 MD)",
)
fig.update_traces(textposition="top center")
show_and_export_fig(fig, "08_risque_vs_rendement_modeles")

### Graphique 9 — Sensibilité du rendement final selon le rendement des 10 MD

In [ ]:
sens_ret = sensitivity.loc[sensitivity["Type"].eq("Rendement final pour 10 MD")]
fig = px.line(
    sens_ret, x="Hypothèse rendement additionnel", y="Rendement final",
    color="Scenario", line_dash="Périmètre", markers=True,
    title="Sensibilité du rendement final selon le rendement des 10 MD",
)
fig.add_hline(y=TARGET_RETURN, line_dash="dash", annotation_text="Cible TSR+4%")
show_and_export_fig(fig, "09_sensibilite_rendement_final")

### Graphique 10 — Montant additionnel requis selon l'hypothèse de rendement

In [ ]:
sens_amt = sensitivity.loc[
    sensitivity["Type"].eq("Montant additionnel requis")
].dropna(subset=["Montant requis"])
fig = px.bar(
    sens_amt, x="Hypothèse rendement additionnel", y="Montant requis",
    color="Scenario", barmode="group", facet_row="Périmètre",
    title="Budget additionnel requis pour atteindre la cible TSR+4%",
)
fig.update_layout(height=620)
show_and_export_fig(fig, "10_budget_additionnel_requis")

print(f"{len(exported_figures)} figures exportées dans {FIGURE_DIR}")

## 14. Export Excel et contrôle final

Le classeur Excel est exporté avec les 15 feuilles attendues : hypothèses, portefeuille, poche optimisable, scénarios APT, rendements requis, allocations, résultats par scénario, impacts, contraintes, Monte Carlo, scoring, sensibilité, recommandation, conclusion.

In [ ]:
from maghrebia_quant.allocation_10md import _build_final_control_multi

figures_ok = len(exported_figures) >= 10 and all(p.exists() for p in exported_figures)
excel_path = export_multi_scenario_analysis(result, EXCEL_PATH)
excel_ok = excel_path.exists()

result["Controle_Final"] = _build_final_control_multi(
    result, result["by_scenario"], CONFIG,
    figures_exported=figures_ok, excel_exported=excel_ok,
)
display(result["Controle_Final"])
display(Markdown(f"Classeur Excel : `{EXCEL_PATH.relative_to(PROJECT_ROOT)}`"))
display(Markdown(f"Figures HTML : `{FIGURE_DIR.relative_to(PROJECT_ROOT)}`"))

## 15. Conclusion finale

Conclusion structurée en 8 points + formulation institutionnelle, conforme à la consigne du PFE.

In [ ]:
display(result["15_Conclusion"])

In [ ]:
central = result["by_scenario"]["APT_Central"]
reco_metrics = central["results_models"].loc[
    central["results_models"]["Model"].eq(central["recommended_model"])
].iloc[0]

control = result["Controle_Final"]
technical_status = control.loc[control["Contrôle"].eq("Technical_Status"), "Status"].iloc[0]
global_status = control.loc[control["Contrôle"].eq("Statut global"), "Status"].iloc[0]

status_lines = []
for scenario in SCENARIO_ORDER:
    scn = result["by_scenario"][scenario]
    reco = scn["results_models"].loc[scn["results_models"]["Model"].eq(scn["recommended_model"])].iloc[0]
    reached = reco["Target_Opt_Reached"] == "YES" and reco["Target_Total_Reached"] == "YES"
    status_lines.append(
        f"- **{scenario}** — modèle recommandé `{scn['recommended_model']}` — "
        f"`Target_Status = {'TARGET_REACHED' if reached else 'TARGET_NOT_REACHED'}` — "
        f"R_additional = {reco['R_additional']:.2%} ; R_opt_final = {reco['R_opt_final']:.2%} ; "
        f"R_total_final = {reco['R_total_final']:.2%} ; Gap_Opt = {reco['Gap_Opt']:+.2%} ; "
        f"Gap_Total = {reco['Gap_Total']:+.2%}."
    )

display(Markdown(
    f"**Technical_Status** = `{technical_status}`  \n"
    f"**Statut global** = `{global_status}`  \n\n"
    + "\n".join(status_lines)
    + (
        "\n\n_Le notebook est techniquement valide. L'objectif TSR + 4 % n'est pas atteint sous "
        "toutes les hypothèses retenues : la non-atteinte traduit un effet de taille et un "
        "rendement marginal requis supérieur aux rendements réalistes disponibles, pas une "
        "défaillance technique de l'analyse._"
    )
))